# Práctica 6 - Preprocesamiento de datos para Machine Learning
**Sistemas de Aprendizaje Automático · Sesión 6**   
**Dataset:** `clientes_ecommerce.csv` · **Modalidad:** individual

## Contexto
Trabajas en una empresa de comercio electrónico que quiere predecir si un cliente comprará el próximo mes. El dataset ya se exploró en las sesiones 4 y 5. Ahora debes transformarlo para que esté listo para entrenar un modelo de clasificación binaria.

**No entrenes modelos en esta práctica.** Completa los bloques `TODO` y responde las preguntas en las celdas Markdown de respuesta.

### Objetivos
- Detectar ausentes y anomalías
- Elegir estrategias de imputación 
- Codificar variables nominales y ordinales
- Comparar escalados
- Separar entrenamiento/prueba
- Construir un `Pipeline` con `ColumnTransformer` 
- Evitar data leakage
- Verificar las matrices resultantes

> Coloca `clientes_ecommerce.csv` en la misma carpeta que este notebook. Usa scikit-learn 1.2 o posterior para `sparse_output=False`.

## Parte 1 - Diagnóstico del dataset
### Ejercicio 1. Carga y estructura
Carga el CSV, muestra 10 filas, dimensiones, tipos de datos, información general y número de registros duplicados.

In [1]:
%pip install scikit-learn

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ------ --------------------------------- 1.3/8.4 MB 5.5 MB/s eta 0:00:02
   ------------ --------------------------- 2.6/8.4 MB 6.6 MB/s eta 0:00:01
   --------------------- ------------------ 4.5/8.4 MB 7.1 MB/s eta 0:00:01
   --------------------------- ------------ 5.8/8.4 MB 7.3 MB/s eta 0:00:01
   -------------------------------- ------- 6.8/8.4 MB 6.7 MB/s eta 0:00:01
   -------------------------------------- - 8.1/8.4 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 6.5 MB/s  0:00:01
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   - -------------------------------------- 1.8/37.4 MB 9.3 MB/s eta 0:00:04
   ---- ----------------------------------- 3.9/37.4 MB 9.5 MB/s eta 0:00:04
   ------ --------------------------------- 5.8/37.4 MB 9.2 MB/s eta 0:00:04
   -------- --------------------

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# TODO: cargar clientes_ecommerce.csv en df
df = pd.read_csv('clientes_ecommerce.csv')
# TODO: mostrar 10 primeras filas
df.head(10)
# TODO: consultar shape, dtypes, info y número de duplicados
df.info()
df.shape
df.dtypes
print('Filas duplicadas:', df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cliente_id          400 non-null    int64  
 1   edad                400 non-null    int64  
 2   antiguedad_meses    395 non-null    float64
 3   ingresos_anuales    396 non-null    float64
 4   gasto_mensual       397 non-null    float64
 5   num_compras         400 non-null    int64  
 6   devoluciones        400 non-null    int64  
 7   dispositivo         400 non-null    str    
 8   zona                400 non-null    str    
 9   nivel_fidelidad     400 non-null    str    
 10  satisfaccion        400 non-null    int64  
 11  compra_proximo_mes  400 non-null    str    
dtypes: float64(3), int64(5), str(4)
memory usage: 37.6 KB
Filas duplicadas: 0


**Responde:** ¿Cuántas muestras y columnas hay? ¿Cuál es el target? ¿Qué columna identifica a cada cliente? ¿Hay filas duplicadas?

**Respuesta:**

- Hay **400 muestras** (filas) y **12 columnas**.
- El **target** es `compra_proximo_mes` (Sí/No): lo que queremos predecir.
- El identificador de cada cliente es `cliente_id`.
- **No hay filas duplicadas** (0 registros duplicados).

### Ejercicio 2. Valores ausentes
Calcula el número de ausentes por columna, el porcentaje y el número de filas con al menos un ausente. Completa la tabla.

In [3]:
df.isna()
# TODO: ausentes por columna
ausentes = df.isna().sum()
ausentes
# TODO: porcentaje de ausentes por columna
porcentaje = df.isna().mean() * 100
porcentaje
# TODO: filas con algún valor ausente
filas_incompletas = df.isna().any(axis=1).sum()
filas_incompletas

np.int64(12)

| Variable | Nº ausentes | Porcentaje |
|---|---:|---:|
| `antiguedad_meses` | 5 | 1,25 % |
| `ingresos_anuales` | 4 | 1,00 % |
| `gasto_mensual` | 3 | 0,75 % |

En total hay 12 filas con al menos un ausente (el 3 % del dataset).

**¿Eliminarías todas las filas incompletas? ¿Por qué?**

Respuesta: No. Aunque son pocas filas, eliminarlas supone perder información útil de otras columnas que sí están completas (y datos de entrenamiento, que ya son pocos: 400). Además, si los ausentes no son completamente aleatorios, borrar esas filas podría sesgar la muestra. Es preferible **imputar** los valores (mediana para las numéricas) dentro del pipeline, ajustando el imputador solo con entrenamiento.

### Ejercicio 3. Edad sospechosa
Localiza edades superiores a 120 años, muestra `cliente_id` y `edad` y consulta mínimo y máximo.

In [4]:
# TODO: mínimo y máximo de edad
print('Edad mínima:', df['edad'].min())
print('Edad máxima:', df['edad'].max())
# TODO: localizar registros con edad > 120
df.loc[df["edad"] > 120, ["cliente_id", "edad"]]

Edad mínima: 18
Edad máxima: 195


,cliente_id,edad
137,20138,195


**Responde:** ¿Qué harías con esa edad? ¿Por qué no deberías cambiar 195 por 95 sin evidencia?

Respuesta: Una edad de 195 años (cliente 20138) es imposible, así que la trataría como un **valor anómalo (error de captura)**: la marcaría como ausente (`NaN`) y la imputaría después con la mediana dentro del pipeline (o, si fuera posible, la corregiría consultando la fuente original). Tras la división ese registro cae en el conjunto de entrenamiento, así que el imputador usa la mediana de entrenamiento para rellenarlo. No debería cambiarla por 95 porque sería **inventar un dato**: el error podría ser 19, 25, 59, 95... y no hay evidencia de cuál era el valor real. Asumir una corrección introduciría un dato falso en el modelo.

## Parte 2 - Variables y división
### Ejercicio 4. Clasificación de columnas
Clasifica las columnas según su naturaleza.

| Tipo | Columnas |
|---|---|
| Numéricas | `edad`, `antiguedad_meses`, `ingresos_anuales`, `gasto_mensual`, `num_compras`, `devoluciones`, `satisfaccion` |
| Categóricas nominales | `dispositivo`, `zona` |
| Categóricas ordinales | `nivel_fidelidad` |
| Identificador | `cliente_id` |
| Target | `compra_proximo_mes` |

**Responde:** ¿Por qué `zona` es nominal y `nivel_fidelidad` ordinal? ¿Por qué no incluirías `cliente_id` en el modelo?

Respuesta: `zona` es **nominal** porque sus categorías son simples etiquetas sin un orden natural (una zona no es "mayor" que otra). `nivel_fidelidad` es **ordinal** porque sus categorías tienen un orden jerárquico claro (Bronce < Plata < Oro < Platino). `cliente_id` no se incluye porque es solo un identificador único: no aporta información sobre el comportamiento del cliente y el modelo podría memorizarlo (sobreajuste) o encontrar patrones espurios, sin capacidad de generalizar a clientes nuevos.

### Ejercicio 5. Separar X e y
Excluye de X el identificador y el target. Codifica `No` como 0 y `Sí` como 1. Comprueba dimensiones y distribución de y.

In [5]:
# X: todas las columnas salvo el identificador y el target
X = df.drop(columns=["cliente_id", "compra_proximo_mes"])
# y: No -> 0, Sí -> 1
y = df["compra_proximo_mes"].map({"No": 0, "Sí": 1})

print('X shape:', X.shape)
print('y shape:', y.shape)
print('y nulos tras el mapeo:', y.isna().sum())
print('y value counts:\n', y.value_counts())
print('y proporciones:\n', y.value_counts(normalize=True).round(3))

X shape: (400, 10)
y shape: (400,)
y nulos tras el mapeo: 0
y value counts:
 compra_proximo_mes
0    246
1    154
Name: count, dtype: int64
y proporciones:
 compra_proximo_mes
0    0.615
1    0.385
Name: proportion, dtype: float64


**¿Cuántas características predictoras iniciales hay?**

Respuesta: **10** características predictoras: `edad`, `antiguedad_meses`, `ingresos_anuales`, `gasto_mensual`, `num_compras`, `devoluciones`, `dispositivo`, `zona`, `nivel_fidelidad` y `satisfaccion`. (12 columnas menos el identificador y el target.)

### Ejercicio 6. División entrenamiento/prueba
Separa 80 % para entrenamiento y 20 % para prueba. Utiliza `random_state=42` y `stratify=y`. Comprueba tamaños.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)
print('y_train:', y_train.shape)
print('y_test: ', y_test.shape)
print('Proporción de compradores en entrenamiento:', round(y_train.mean(), 3))
print('Proporción de compradores en prueba:       ', round(y_test.mean(), 3))

X_train: (320, 10)
X_test:  (80, 10)
y_train: (320,)
y_test:  (80,)
Proporción de compradores en entrenamiento: 0.384
Proporción de compradores en prueba:        0.388


**Responde:** ¿Cuántas muestras hay en cada conjunto? ¿Para qué sirve `stratify=y`? ¿Por qué dividimos antes de ajustar transformaciones?

Respuesta: Con 400 muestras y `test_size=0.2`, hay **320 muestras de entrenamiento** y **80 de prueba**. `stratify=y` hace que la proporción de clases del target se mantenga prácticamente igual en entrenamiento y prueba: en el dataset completo hay un 38,5 % de compradores (154 Sí frente a 246 No) y tras la división queda un 38,4 % en entrenamiento y un 38,8 % en prueba. Es importante cuando las clases no están equilibradas, como aquí. Dividimos **antes** de ajustar cualquier transformación para evitar **data leakage**: si calculáramos medianas, medias o categorías con todos los datos, la información del conjunto de prueba se filtraría al entrenamiento y la evaluación sería demasiado optimista.

## Parte 3 - Imputación y encoding
### Ejercicio 7. Validación de edad
Sobre **copias** de `X_train` y `X_test`, sustituye edades superiores a 120 por `np.nan`. No imputes todavía: eso se hará en el pipeline. El umbral es una regla didáctica de validación del dominio.

In [7]:
X_train = X_train.copy()
X_test = X_test.copy()

# Marcar edades > 120 como NaN en ambos conjuntos (mask convierte a float y deja NaN)
X_train["edad"] = X_train["edad"].mask(X_train["edad"] > 120)
X_test["edad"] = X_test["edad"].mask(X_test["edad"] > 120)

# Contar ausentes en edad en entrenamiento y prueba
print('Ausentes en edad (entrenamiento):', X_train["edad"].isna().sum())
print('Ausentes en edad (prueba):       ', X_test["edad"].isna().sum())

Ausentes en edad (entrenamiento): 1
Ausentes en edad (prueba):        0


### Ejercicio 8. Estrategias de imputación
Utilizaremos mediana para numéricas y moda para categóricas. Crea ambos objetos `SimpleImputer` **sin ajustarlos todavía**.

In [8]:

"""strategy:
- Media: mean
- Mediana: median
- Moda: most_frequent
- Valor constante: constant => fill_value=x
"""
# TODO: imputer_numerico = SimpleImputer(...)
imputer_numerico = SimpleImputer(strategy='median')
# TODO: imputer_categorico = SimpleImputer(...)
imputer_categorico = SimpleImputer(strategy='most_frequent')

| Variable | Estrategia | Justificación |
|---|---|---|
| edad | Mediana | Numérica; tras marcar el 195 como `NaN`, la mediana es robusta frente a valores extremos o anomalías. |
| antiguedad_meses | Mediana | Numérica, posible asimetría y valores extremos; la mediana no se ve afectada por ellos. |
| ingresos_anuales | Mediana | Los ingresos suelen estar sesgados a la derecha; la media se distorsionaría con ingresos muy altos. |
| gasto_mensual | Mediana | Igual que los ingresos: distribución típicamente asimétrica con valores extremos. |
| dispositivo | Moda (`most_frequent`) | Categórica nominal: no se puede calcular una mediana; se usa la categoría más frecuente. |
| zona | Moda (`most_frequent`) | Categórica nominal: se imputa con la zona más frecuente en entrenamiento. |

**¿Por qué no ajustar el imputador con entrenamiento y prueba a la vez?**

Respuesta: Porque la mediana o la moda calculadas con todos los datos incluirían información del conjunto de prueba (data leakage). El imputador debe aprender los valores solo con **entrenamiento** (`fit`) y luego aplicarlos a prueba (`transform`), como ocurriría con datos nuevos en producción, donde no conocemos su distribución de antemano.

### Ejercicio 9. One-Hot Encoding
Prueba `OneHotEncoder` con un DataFrame independiente. Configura `handle_unknown="ignore"` y `sparse_output=False`. Muestra matriz y nombres de columnas.

In [9]:
ejemplo = pd.DataFrame({
    "dispositivo": ["Móvil", "Ordenador", "Tablet", "Móvil"]
})

# Encoder One-Hot: ignora categorías desconocidas y devuelve matriz densa
encoder_nominal = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
matriz_onehot = encoder_nominal.fit_transform(ejemplo)

print(matriz_onehot)
print(encoder_nominal.get_feature_names_out())

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
['dispositivo_Móvil' 'dispositivo_Ordenador' 'dispositivo_Tablet']


**Responde:** ¿Cuántas columnas se generan? ¿Qué representan? ¿Por qué no codificar Móvil=0, Ordenador=1 y Tablet=2 como categorías ordenadas?

Respuesta: Se generan **3 columnas**, una por categoría (`dispositivo_Móvil`, `dispositivo_Ordenador`, `dispositivo_Tablet`). Cada columna es una variable binaria que vale 1 si la muestra pertenece a esa categoría y 0 si no. No conviene codificar Móvil=0, Ordenador=1, Tablet=2 porque los números **introducen un orden y una distancia artificiales** (el modelo interpretaría que Tablet es "más" que Ordenador o que la distancia Móvil-Tablet es el doble que Móvil-Ordenador), cuando los dispositivos son categorías nominales sin jerarquía.

### Ejercicio 10. Codificación ordinal
Configura `OrdinalEncoder` para el orden Bronce, Plata, Oro, Platino. Comprueba su funcionamiento con un ejemplo independiente.

In [10]:
ejemplo_fidelidad = pd.DataFrame({
    "nivel_fidelidad": ["Oro", "Bronce", "Platino", "Plata"]
})

encoder_ordinal = OrdinalEncoder(categories=[["Bronce", "Plata", "Oro", "Platino"]])
codigos = encoder_ordinal.fit_transform(ejemplo_fidelidad)

print(pd.DataFrame({
    "nivel_fidelidad": ejemplo_fidelidad["nivel_fidelidad"],
    "codigo": codigos.ravel()
}))

  nivel_fidelidad  codigo
0             Oro     2.0
1          Bronce     0.0
2         Platino     3.0
3           Plata     1.0


**Responde:** ¿Qué representan los códigos? ¿La distancia entre niveles consecutivos es necesariamente igual?

Respuesta: Los códigos representan la **posición en la jerarquía**: Bronce=0, Plata=1, Oro=2, Platino=3 (en el ejemplo: Oro→2, Bronce→0, Platino→3, Plata→1). Mantienen el orden entre niveles, pero **la distancia entre niveles consecutivos no tiene por qué ser igual**: pasar de Bronce a Plata no equivale necesariamente, en valor o comportamiento de compra, a pasar de Oro a Platino. La codificación ordinal asume implícitamente una separación uniforme, lo que es una simplificación.

## Parte 4 - Escalado
### Ejercicio 11. Min-Max frente a Z-score
Aplica ambos escaladores al siguiente ejemplo. Redondea a tres decimales y completa la tabla.

In [11]:
datos = pd.DataFrame({"valor": [10, 20, 30, 40, 50]})

# Crear MinMaxScaler y StandardScaler
minmax = MinMaxScaler()
standard = StandardScaler()

# Transformar datos con cada uno
datos_minmax = minmax.fit_transform(datos)
datos_standard = standard.fit_transform(datos)

# Tabla comparativa
tabla = pd.DataFrame({
    "Original": datos["valor"],
    "Min-Max": datos_minmax.ravel().round(3),
    "StandardScaler": datos_standard.ravel().round(3)
})
print(tabla)
print('Media tras estandarizar:', round(datos_standard.mean(), 3))
print('Desviación típica tras estandarizar:', round(datos_standard.std(), 3))

   Original  Min-Max  StandardScaler
0        10     0.00          -1.414
1        20     0.25          -0.707
2        30     0.50           0.000
3        40     0.75           0.707
4        50     1.00           1.414
Media tras estandarizar: 0.0
Desviación típica tras estandarizar: 1.0


| Original | Min-Max | StandardScaler |
|---:|---:|---:|
| 10 | 0.000 | -1.414 |
| 20 | 0.250 | -0.707 |
| 30 | 0.500 | 0.000 |
| 40 | 0.750 | 0.707 |
| 50 | 1.000 | 1.414 |

**Responde:** ¿Qué rango da Min-Max en este ejemplo? ¿Qué valor toma la media tras estandarizar? ¿Pueden aparecer valores negativos? ¿Qué escalado elegirías para un algoritmo basado en distancias y por qué?

Respuesta: Min-Max da el rango **[0, 1]**. Tras estandarizar, la **media es 0** (y la desviación típica 1). Con StandardScaler **sí aparecen valores negativos** (los que están por debajo de la media); con Min-Max no, salvo que haya datos nuevos fuera del rango de entrenamiento. Para un algoritmo basado en distancias (k-NN, k-means, SVM...) elegiría **StandardScaler**, ya que pone todas las variables en la misma escala (media 0, desviación 1) y es menos sensible a valores extremos que Min-Max, donde un solo outlier comprime al resto de datos en un rango pequeño. Min-Max también es válido si se necesita un rango acotado en [0, 1] y no hay outliers marcados.

### Ejercicio 12. Variables que escalaremos
Para esta práctica trataremos `satisfaccion` como numérica, aunque conceptualmente es una escala ordinal de 1 a 5. Es una simplificación que deberá revisarse en otros modelos.

In [12]:
numericas = [
    "edad", "antiguedad_meses", "ingresos_anuales",
    "gasto_mensual", "num_compras", "devoluciones", "satisfaccion"
]
nominales = ["dispositivo", "zona"]
ordinales = ["nivel_fidelidad"]

# Comprobar que las columnas existen en X_train
todas = numericas + nominales + ordinales
print('Columnas que faltan en X_train:', [c for c in todas if c not in X_train.columns])
print('Columnas de X_train sin asignar:', [c for c in X_train.columns if c not in todas])

# Escalas de las variables numéricas (para ver cuáles son más distintas)
print(X_train[numericas].describe().loc[["min", "max", "mean", "std"]].round(2).to_string())

Columnas que faltan en X_train: []
Columnas de X_train sin asignar: []
       edad  antiguedad_meses  ingresos_anuales  gasto_mensual  num_compras  devoluciones  satisfaccion
min   18.00              2.00           9000.00           5.77         2.00          0.00          1.00
max   75.00            120.00          82925.37         260.00        17.00          5.00          5.00
mean  46.13             59.94          36603.50          68.22         8.24          1.21          3.29
std   16.47             34.02          14205.82          38.12         2.73          1.13          1.14


**Responde:** ¿Qué variables tienen escalas más distintas? ¿Es obligatorio escalar para cualquier algoritmo? ¿Qué particularidad tiene `satisfaccion`?

Respuesta: Las escalas más distintas se dan entre `ingresos_anuales` (de unos 9.000 a unos 83.000) y `gasto_mensual` (de unos 6 a 260), frente a `devoluciones` (0-5) y `satisfaccion` (1-5), que son valores muy pequeños; `edad` (18-75 tras limpiar el 195) y `antiguedad_meses` (1-120) quedan en un rango intermedio. Sin escalar, `ingresos_anuales` dominaría cualquier cálculo de distancias. **No es obligatorio escalar** para cualquier algoritmo: los modelos basados en árboles (árbol de decisión, Random Forest, Gradient Boosting) no se ven afectados por la escala, mientras que los basados en distancias o gradientes (k-NN, SVM, regresión logística, redes neuronales) sí lo necesitan. La particularidad de `satisfaccion` es que conceptualmente es una **escala ordinal de 1 a 5**, aunque aquí la tratamos como numérica; es una simplificación que supone distancias iguales entre puntos de la escala y que habría que revisar en otros modelos.

## Parte 5 - Pipeline completo
### Ejercicio 13. Pipeline numérico
Construye un `Pipeline` que impute con mediana y después estandarice.

In [13]:
pipeline_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

**¿Por qué imputamos antes de escalar en este pipeline?**

Respuesta: Porque el escalador necesita valores completos para calcular correctamente la media y la desviación típica (y no puede trabajar con `NaN`). Además, imputar primero hace que los valores imputados se tengan en cuenta al calcular los parámetros de escalado, de modo que todos los datos queden en la misma escala. Si se escalara antes, las estadísticas se calcularían ignorando los ausentes y luego habría que imputar con valores en otra escala.

### Ejercicio 14. Pipeline nominal
Imputa con moda y después aplica One-Hot Encoding (`handle_unknown="ignore"`, `sparse_output=False`).

In [14]:
pipeline_nominal = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

### Ejercicio 15. Pipeline ordinal
Imputa con moda y codifica Bronce → Plata → Oro → Platino. Gestiona categorías desconocidas con `handle_unknown="use_encoded_value"` y `unknown_value=-1`.

In [15]:
pipeline_ordinal = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[["Bronce", "Plata", "Oro", "Platino"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

### Ejercicio 16. ColumnTransformer
Combina los tres pipelines para aplicar cada transformación únicamente a sus columnas.

In [16]:
preprocesador = ColumnTransformer(transformers=[
    ("num", pipeline_numerico, numericas),
    ("nom", pipeline_nominal, nominales),
    ("ord", pipeline_ordinal, ordinales)
])

### Ejercicio 17. Transformar entrenamiento y prueba
Ajusta el preprocesador **solo** con entrenamiento y aplícalo a prueba sin reajustarlo.

In [17]:
# fit solo con entrenamiento: aprende medianas, modas, medias, desviaciones y categorías
X_train_preparado = preprocesador.fit_transform(X_train)
# transform en prueba: aplica lo aprendido con entrenamiento, sin reajustar
X_test_preparado = preprocesador.transform(X_test)

**Responde:** ¿Por qué no usar `fit_transform()` en prueba? ¿Qué aprende `fit()`? ¿Qué ocurre si aparece una categoría nominal nueva en prueba?

Respuesta: No se usa `fit_transform()` en prueba porque reajustaría el preprocesador con datos de prueba (aprendería medianas, medias y categorías de ese conjunto), lo que provoca **data leakage** y además transformaría entrenamiento y prueba con parámetros distintos. `fit()` aprende, solo con entrenamiento, las **medianas** y **modas** de los imputadores, la **media y desviación típica** del escalador y las **categorías** de los codificadores. Si aparece una categoría nominal nueva en prueba, gracias a `handle_unknown="ignore"` el One-Hot Encoder no falla: pone un 0 en todas las columnas de esa variable (no activa ninguna categoría conocida). En la variable ordinal, `unknown_value=-1` asigna el código -1 a las categorías desconocidas.

## Parte 6 - Validación e informe
### Ejercicio 18. Verificar matrices
Comprueba tamaños, ausentes, tipos numéricos, correspondencia con las etiquetas y nombres de características.

In [18]:
# Tamaños
print('X_train_preparado:', X_train_preparado.shape)
print('X_test_preparado: ', X_test_preparado.shape)

# Ausentes
print('NaN en entrenamiento:', np.isnan(X_train_preparado).sum())
print('NaN en prueba:       ', np.isnan(X_test_preparado).sum())

# Tipo y nombres de características
print('dtype:', X_train_preparado.dtype)
print(preprocesador.get_feature_names_out())

# Correspondencia con las etiquetas
print('Filas train == len(y_train):', X_train_preparado.shape[0] == len(y_train))
print('Filas test  == len(y_test): ', X_test_preparado.shape[0] == len(y_test))

# Mismo número de columnas
print('Mismas columnas en train y test:', X_train_preparado.shape[1] == X_test_preparado.shape[1])

X_train_preparado: (320, 16)
X_test_preparado:  (80, 16)
NaN en entrenamiento: 0
NaN en prueba:        0
dtype: float64
['num__edad' 'num__antiguedad_meses' 'num__ingresos_anuales'
 'num__gasto_mensual' 'num__num_compras' 'num__devoluciones'
 'num__satisfaccion' 'nom__dispositivo_Móvil' 'nom__dispositivo_Ordenador'
 'nom__dispositivo_Tablet' 'nom__zona_Centro' 'nom__zona_Este'
 'nom__zona_Norte' 'nom__zona_Oeste' 'nom__zona_Sur'
 'ord__nivel_fidelidad']
Filas train == len(y_train): True
Filas test  == len(y_test):  True
Mismas columnas en train y test: True


**Responde:** ¿Por qué aumentaron las columnas? ¿Coinciden las características en ambos conjuntos? ¿Quedan ausentes? ¿Están alineadas las etiquetas?

Respuesta: Las columnas pasaron de 10 a **16** porque el One-Hot Encoding convierte cada variable nominal en **una columna binaria por categoría**: 7 numéricas + 3 de `dispositivo` (Móvil, Ordenador, Tablet) + 5 de `zona` (Centro, Este, Norte, Oeste, Sur) + 1 ordinal (`nivel_fidelidad`) = 16. Las características **coinciden en ambos conjuntos** (mismo número de columnas y mismo orden), porque el preprocesador se ajustó solo con entrenamiento y se aplicó a prueba con `transform`. **No quedan ausentes** (0 `NaN` en ambas matrices), ya que los imputadores los han rellenado. Las **etiquetas están alineadas**: el número de filas de cada matriz coincide con `len(y_train)` y `len(y_test)`, y el orden de las filas se mantiene.

### Ejercicio 19. Reconstruir DataFrames
Convierte las matrices preparadas en `df_train_preparado` y `df_test_preparado`. Usa `preprocesador.get_feature_names_out()` y conserva los índices de X_train/X_test.

In [19]:
columnas_finales = preprocesador.get_feature_names_out()

df_train_preparado = pd.DataFrame(X_train_preparado, columns=columnas_finales, index=X_train.index)
df_test_preparado = pd.DataFrame(X_test_preparado, columns=columnas_finales, index=X_test.index)

display(df_train_preparado.head())
display(df_test_preparado.head())

,num__edad,num__antiguedad_meses,num__ingresos_anuales,num__gasto_mensual,num__num_compras,num__devoluciones,num__satisfaccion,nom__dispositivo_Móvil,nom__dispositivo_Ordenador,nom__dispositivo_Tablet,nom__zona_Centro,nom__zona_Este,nom__zona_Norte,nom__zona_Oeste,nom__zona_Sur,ord__nivel_fidelidad
360,-0.068723,-1.095565,0.651045,-1.086877,0.277945,-0.185205,-0.254293,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0
356,-0.677904,1.129874,1.422884,-1.092939,0.277945,-1.069767,-0.254293,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
138,0.845048,-1.540653,0.907575,0.537957,-0.457116,-1.069767,1.495680,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
42,-1.408922,0.447406,0.476223,0.303390,1.380537,-1.069767,1.495680,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
216,-0.495150,1.456272,-0.493983,1.466735,0.645476,-1.069767,1.495680,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


,num__edad,num__antiguedad_meses,num__ingresos_anuales,num__gasto_mensual,num__num_compras,num__devoluciones,num__satisfaccion,nom__dispositivo_Móvil,nom__dispositivo_Ordenador,nom__dispositivo_Tablet,nom__zona_Centro,nom__zona_Este,nom__zona_Norte,nom__zona_Oeste,nom__zona_Sur,ord__nivel_fidelidad
388,0.053113,-1.540653,-0.994595,-0.887364,1.013006,1.583919,-0.254293,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
302,-0.495150,1.040856,-0.185324,-0.504414,-0.089586,-1.069767,-0.254293,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
21,1.697902,-1.036220,0.611217,-0.253507,-0.457116,1.583919,-0.254293,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0
368,0.784130,-1.095565,0.373354,-0.176812,-1.559708,1.583919,0.620694,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0
378,-0.312396,0.269371,-1.092628,1.067445,1.013006,-1.069767,1.495680,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
